In [55]:
import pandas as pd

In [66]:
df_occupations_with_sectors = pd.read_csv("occupations_with_sectors.csv")
df_other_services = df_occupations_with_sectors[df_occupations_with_sectors['SUBSECTOR']=="Other services"]
df_other_services.to_csv("occupations_with_sectors_others.csv",index=False)

In [ ]:
# enrich occupation_with_sectors with to alt values
df_manually_matched = pd.read_csv("D:\LMIS\Job matching\manual_matched\manual_alt1_score.csv")
count = 0
for index, row in df_manually_matched.iterrows():
    alt = str(row["Alt_1"]).strip().lower() if pd.notna(row["Alt_1"]) else None
    preferred = (
        str(row["PREFERREDLABEL_1"]).strip()
        if pd.notna(row["PREFERREDLABEL_1"])
        else None
    )
    scope = str(row["SCOPE"]).strip() if pd.notna(row["SCOPE"]) else None

    # Skip invalid rows
    if (
        scope != "IN"
        or not preferred
        or not alt
    ):
        continue

    # Find matching occupation rows
    mask = (
        df_occupations_with_sectors["PREFERREDLABEL"]
        .astype(str)
        .str.strip()
        == preferred
    )

    matched_indexes = df_occupations_with_sectors[mask].index

    for matched_index in matched_indexes:

        current_altlabels = df_occupations_with_sectors.at[
            matched_index,
            "ALTLABELS"
        ]

        # Handle empty/null ALTLABELS
        if pd.isna(current_altlabels) or str(current_altlabels).strip() == "":
            existing_labels = []
        else:
            existing_labels = [
                label.strip().lower()
                for label in str(current_altlabels).split("\n")
                if label.strip().lower()
            ]

        # Append only if not already present
        if alt not in existing_labels:

            existing_labels.append(alt)

            # Save back as newline separated values
            df_occupations_with_sectors.at[
                matched_index,
                "ALTLABELS"
            ] = "\n".join(existing_labels)

            count += 1

print("Enriched altlabels:", count)

# Save updated file
df_occupations_with_sectors.to_csv(
    "occupations_with_sectors.csv",
    index=False
)

print("Saved enriched file.")

In [70]:
def clean_and_lower_altlabels(altlabels):

    # Handle null values
    if pd.isna(altlabels):
        return altlabels

    labels = [
        label.strip().lower()
        for label in str(altlabels).split("\n")
        if label.strip()
    ]

    # Remove duplicates
    unique_labels = []
    seen = set()

    for label in labels:

        if label not in seen:
            seen.add(label)
            unique_labels.append(label)

    return "\n".join(unique_labels)


# Apply cleaning
df_occupations_with_sectors["ALTLABELS"] = (
    df_occupations_with_sectors["ALTLABELS"]
    .apply(clean_and_lower_altlabels)
)

# Save cleaned dataframe
df_occupations_with_sectors.to_csv(
    "occupations_with_sectors.csv",
    index=False
)

print("Converted ALTLABELS to lowercase and removed duplicates.")

Converted ALTLABELS to lowercase and removed duplicates.


In [57]:
#mapp sector_id to occupation_with_sector
import json
import pandas as pd
sector_ids = None
sub_sector_ids = None
with open("sector_sub_sector_ids.json","r") as file:
    ids = json.load(file)
    sector_ids = ids['sectors']
    sub_sector_ids = ids['sub_sectors']

sectors_id_map = {sector['name']:sector['id'] for sector in sector_ids}
sub_sector_id_map = {sub_sector['name']:sub_sector['id'] for sub_sector in sub_sector_ids}
df_occupations_with_sectors = pd.read_csv("occupations_with_sectors.csv")
# df_occupations_with_sectors[]
df_occupations_with_sectors['SECTORID'] = df_occupations_with_sectors['SECTOR'].map(sectors_id_map)
df_occupations_with_sectors['SUBSECTORID'] = df_occupations_with_sectors['SUBSECTOR'].map(sub_sector_id_map)
# df_occupations_with_sectors.head()
df_occupations_with_sectors.to_csv("occupations_with_sectors.csv",index=False)


In [71]:
#merge occupation and skills and keep only important files
import pandas as pd


def rename_columns(df, prefix):
    """Rename standard metadata columns with a prefix."""
    return df.rename(columns={
        "PREFERREDLABEL": f"{prefix}_PREFERREDLABEL",
        "ALTLABELS": f"{prefix}_ALTLABELS",
        "DESCRIPTION": f"{prefix}_DESCRIPTION"
    })


def validate_columns(df, required_cols, name):
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")


def combine_occupation_skill(
    occupation_skill_path,
    skills_path,
    occupation_path,
    output_path=None
):
    # Load CSVs
    df_relation = pd.read_csv(occupation_skill_path, dtype=str)
    df_skills = pd.read_csv(skills_path, dtype=str)
    df_occupation = pd.read_csv(occupation_path, dtype=str)

    # Validate
    validate_columns(df_relation, ["OCCUPATIONID", "SKILLID"], "relation file")
    validate_columns(df_skills, ["ID"], "skills file")
    validate_columns(df_occupation, ["ID"], "occupation file")

    # Rename
    df_skills = rename_columns(df_skills, "SKILL")
    df_occupation = rename_columns(df_occupation, "OCCUPATION")

    # Merge occupation
    df_merge = df_relation.merge(
        df_occupation[
            [
                "ID",
                "OCCUPATION_PREFERREDLABEL",
                "OCCUPATION_ALTLABELS",
                "OCCUPATION_DESCRIPTION",
                "SECTOR",
                "SECTORID",
                "SUBSECTOR",
                "SUBSECTORID"
            ]
        ],
        how="left",
        left_on="OCCUPATIONID",
        right_on="ID"
    ).drop(columns=["ID"])

    # Merge skills
    df_merge = df_merge.merge(
        df_skills[
            [
                "ID",
                "SKILL_PREFERREDLABEL",
                "SKILL_ALTLABELS",
                "SKILL_DESCRIPTION"
            ]
        ],
        how="left",
        left_on="SKILLID",
        right_on="ID"
    ).drop(columns=["ID"])

    # Final order
    column_order = [
        "OCCUPATIONID",
        "OCCUPATION_PREFERREDLABEL",
        "OCCUPATION_ALTLABELS",
        "OCCUPATION_DESCRIPTION",
        # "SECTOR",
        # "SECTORID",
        # "SUBSECTOR",
        # "SUBSECTORID",
        "SKILLID",
        "SKILL_PREFERREDLABEL",
        "SKILL_ALTLABELS",
        "SKILL_DESCRIPTION",
        "RELATIONTYPE"
    ]

    df_merge = df_merge[column_order]

    # Optional save
    if output_path:
        df_merge.to_csv(output_path, index=False)

    return df_merge


# Usage
df = combine_occupation_skill(
    "occupation_to_skill_relations.csv",
    "skills.csv",
    "occupations_with_sectors.csv",
    output_path="combined_occupation_to_skill.csv"
)

print(df.head())

               OCCUPATIONID      OCCUPATION_PREFERREDLABEL  \
0  687671c1ebd2bf665349e10f              musical conductor   
1  687671c1ebd2bf665349e10d                 music director   
2  687671c1ebd2bf665349e110      choirmaster/choirmistress   
3  687671c1ebd2bf665349e10e                       musician   
4  687671c0ebd2bf665349dd9e  correctional services manager   

                                OCCUPATION_ALTLABELS  \
0  conductor, vocal group\nvocal group conductor\...   
1  orchestra conductor\nmusic conductor\nband dir...   
2  choirmistress\nchoir director\nchoral director...   
3  instrumental musician, wind instrument\nnight ...   
4  chief prison officer\ncorrectional institution...   

                              OCCUPATION_DESCRIPTION  \
0  Musical conductors lead ensembles of musicians...   
1  Music directors lead musical groups such as or...   
2  Choirmasters/choirmistresses manage various as...   
3  Musicians perform a vocal or musical part that...   
4  Correct

In [59]:
import pandas as pd


In [72]:
df_combined_occupation = pd.read_csv("combined_occupation_to_skill.csv")
df_skill_hierarchy = pd.read_csv("D:\LMIS\Job matching\ethiopian_taxonomy\skill_hierarchy.csv")
df_occupations_hierarchy = pd.read_csv("D:\LMIS\Job matching\ethiopian_taxonomy\occupation_hierarchy.csv")
df_skill_groups = pd.read_csv("D:\LMIS\Job matching\ethiopian_taxonomy\skill_groups.csv")
df_occupation_groups = pd.read_csv("D:\LMIS\Job matching\ethiopian_taxonomy\occupation_groups.csv")
df_skills = pd.read_csv("D:\LMIS\Job matching\ethiopian_taxonomy\skills.csv")

In [ ]:
# Adding SkillGroup and SkillGroup label to COMBINED FILE
skill_found = 0
skill_not_found = 0
df_combined_occupation["SKILLGROUPID"] = ""
df_combined_occupation["SKILLGROUPLABEL"] = ""
for index,row in df_combined_occupation.iterrows():
    skill_id = row['SKILLID']
    skill_hierarchy = df_skill_hierarchy[(df_skill_hierarchy['CHILDID']==skill_id)&(df_skill_hierarchy['PARENTOBJECTTYPE']=="skillgroup")]
    if not skill_hierarchy.empty:
        group_id = skill_hierarchy['PARENTID'].values
        df_combined_occupation.at[index,'SKILLGROUPID'] = group_id[0]
        skill_group = df_skill_groups[df_skill_groups['ID']==group_id[0]]
        df_combined_occupation.at[index,'SKILLGROUPLABEL'] = skill_group['PREFERREDLABEL'].values
        skill_found +=1
    else:
        skill_not_found +=1

In [42]:
#adding skillsGroup and skillsLable to SKILLS file
skill_found = 0
skill_not_found = 0
df_skills["SKILLGROUPID"] = ""
df_skills["SKILLGROUPLABEL"] = ""
for index,row in df_skills.iterrows():
    skill_id = row['ID']
    skill_hierarchy = df_skill_hierarchy[(df_skill_hierarchy['CHILDID']==skill_id)&(df_skill_hierarchy['PARENTOBJECTTYPE']=="skillgroup")]
    if not skill_hierarchy.empty:
        group_id = skill_hierarchy['PARENTID'].values
        df_skills.at[index,'SKILLGROUPID'] = group_id[0]
        skill_group = df_skill_groups[df_skill_groups['ID']==group_id[0]]
        df_skills.at[index,'SKILLGROUPLABEL'] = skill_group['PREFERREDLABEL'].values
        skill_found +=1
    else:
        skill_not_found +=1

In [43]:
df_skills.head()
df_skills_finals = df_skills[['ID','PREFERREDLABEL','SKILLGROUPLABEL','SKILLGROUPID']]
df_skills_finals.rename(columns={'ID':"SKILLID"},inplace=True)
df_skills_finals.to_csv("skills_with_group.csv")
df_skills_finals.head()

,SKILLID,PREFERREDLABEL,SKILLGROUPLABEL,SKILLGROUPID
0,687671b4ebd2bf665349a631,manage musical staff,supervising a team or group,687671b3ebd2bf665349a59d
1,687671b4ebd2bf665349a632,supervise correctional procedures,directing operational activities,687671b3ebd2bf665349a56b
2,687671b4ebd2bf665349a633,apply anti-oppressive practices,protecting and enforcing,687671b3ebd2bf665349a62c
3,687671b4ebd2bf665349a634,control compliance of railway vehicles regulat...,ensuring compliance with legislation,687671b3ebd2bf665349a4eb
4,687671b4ebd2bf665349a635,identify available services,assisting people to access services,687671b3ebd2bf665349a4d3


In [73]:
# merge occupation and occupation group
occupation_found = 0
occupation_not_found = 0
df_occupations_with_sectors["OCCUPATIONGROUPID"] = ""
df_occupations_with_sectors["OCCUPATIONGROUPLABEL"] = ""
for index,row in df_occupations_with_sectors.iterrows():
    occupation_id = row['ID']
    occupation_hierarchy = df_occupations_hierarchy[(df_occupations_hierarchy['CHILDID']==occupation_id)&(df_occupations_hierarchy['PARENTOBJECTTYPE']=="iscogroup")]
    if not occupation_hierarchy.empty:
        group_id = occupation_hierarchy['PARENTID'].values
        df_occupations_with_sectors.at[index,'OCCUPATIONGROUPID'] = group_id[0]
        occupation_group = df_occupation_groups[df_occupation_groups['ID']==group_id[0]]
        df_occupations_with_sectors.at[index,'OCCUPATIONGROUPLABEL'] = occupation_group['PREFERREDLABEL'].values
        occupation_found +=1
    else:
        occupation_not_found +=1

In [75]:
df_occupation_with_group = df_occupations_with_sectors[['ID','PREFERREDLABEL','ALTLABELS','OCCUPATIONGROUPLABEL','OCCUPATIONGROUPID',"SECTOR","SECTORID","SUBSECTOR","SUBSECTORID",]]
df_occupation_with_group.rename(columns={'ID':"OCCUPATIONID"},inplace=True)
df_occupation_with_group.to_csv("occupations_with_group.csv")
df_occupation_with_group.head()

,OCCUPATIONID,PREFERREDLABEL,ALTLABELS,OCCUPATIONGROUPLABEL,OCCUPATIONGROUPID,SECTOR,SECTORID,SUBSECTOR,SUBSECTORID
0,a6de27fcdc453b8ca2a0105c,auditor general,chief government auditor\ncomptroller and audi...,Senior government officials,687671b1ebd2bf665349a136,Service,b5ab236c-527f-4277-b21b-6cb432afe3e3,Accounting audit and adminstrative works,63b31123-a66f-498d-b586-59337b7f0c54
1,d2985a88c2d72a5af11777c6,head of government,chancellor\ndeputy president\nfirst minister\n...,Senior government officials,687671b1ebd2bf665349a136,Service,b5ab236c-527f-4277-b21b-6cb432afe3e3,Other services,262f28b9-a154-4fa2-85e3-ab14a3f8bfc0
2,986a045206e00d40418cdcc8,senior judiciary member,attorney general\nchief justice\nchief public ...,Senior government officials,687671b1ebd2bf665349a136,Service,b5ab236c-527f-4277-b21b-6cb432afe3e3,Other services,262f28b9-a154-4fa2-85e3-ab14a3f8bfc0
3,751bdeba0deb0cfb49ba00d0,senior local government official,"chief executive, local authority\ncouncillor\n...",Senior government officials,687671b1ebd2bf665349a136,Service,b5ab236c-527f-4277-b21b-6cb432afe3e3,Other services,262f28b9-a154-4fa2-85e3-ab14a3f8bfc0
4,d7958e7d5771a01c1b89b2c1,village chief,community chief\nhead of village\nlocal chieft...,Traditional chiefs and heads of village,687671b1ebd2bf665349a137,Service,b5ab236c-527f-4277-b21b-6cb432afe3e3,Other services,262f28b9-a154-4fa2-85e3-ab14a3f8bfc0


In [65]:
df_occupation_with_group.info()

<class 'pandas.DataFrame'>
RangeIndex: 3143 entries, 0 to 3142
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   OCCUPATIONID          3143 non-null   str  
 1   PREFERREDLABEL        3143 non-null   str  
 2   OCCUPATIONGROUPLABEL  3143 non-null   str  
 3   OCCUPATIONGROUPID     3143 non-null   str  
 4   SECTOR                3143 non-null   str  
 5   SECTORID              3143 non-null   str  
 6   SUBSECTOR             3143 non-null   str  
 7   SUBSECTORID           3052 non-null   str  
dtypes: str(8)
memory usage: 196.6 KB


In [64]:
# df_occupation_with_group[df_occupation_with_group['SECTOR'].isna()]
df = df_occupations_with_sectors
mask = (
        df["SECTOR"].isna() | (df["SECTOR"] == "") |
        df["SUBSECTOR"].isna() | (df["SUBSECTOR"] == "")
    )
df[mask].info()

<class 'pandas.DataFrame'>
RangeIndex: 0 entries
Data columns (total 20 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   ID                       0 non-null      str  
 1   ORIGINURI                0 non-null      str  
 2   UUIDHISTORY              0 non-null      str  
 3   OCCUPATIONGROUPCODE      0 non-null      str  
 4   CODE                     0 non-null      str  
 5   DEFINITION               0 non-null      str  
 6   SCOPENOTE                0 non-null      str  
 7   REGULATEDPROFESSIONNOTE  0 non-null      str  
 8   OCCUPATIONTYPE           0 non-null      str  
 9   ISLOCALIZED              0 non-null      bool 
 10  PREFERREDLABEL           0 non-null      str  
 11  ALTLABELS                0 non-null      str  
 12  DESCRIPTION              0 non-null      str  
 13  CONFIDENCE               0 non-null      str  
 14  SECTOR                   0 non-null      str  
 15  SUBSECTOR                0 no